# Trimmed Simple HALE - Gust Response

This notebook trims a simple HALE aircraft, then flies it through a 1-minus-cosine gust as a free-flying body.

## Imports

In [ ]:
import matplotlib.pyplot as plt
from jax import numpy as jnp

from flapjax.aero.flowfields import OneMinusCosineFlowField
from flapjax.models.simple_hale.simple_hale import generate_simple_hale

## Case parameters

Define the free-stream conditions, gust properties, and simulation time.

In [ ]:
u_inf_mag: float = 10.0
gust_intensity: float = 0.2
gust_length: float = 1.0 * u_inf_mag  # 1 second gust duration
physical_time: float = 10.0

sigma_wing: float = 1.5  # stiffness multiplier for the wing structure

## Build the gust flowfield

A one-minus-cosine vertical gust positioned upstream of the aircraft, with `relative_motion=True`
for the trim solve (clamped aircraft, moving air).

In [ ]:
flowfield = OneMinusCosineFlowField(
    u_inf=jnp.array((u_inf_mag, 0.0, 0.0)),
    rho=1.225,
    relative_motion=True,
    gust_length=gust_length,
    gust_amplitude=gust_intensity * u_inf_mag,
    gust_x0=jnp.array((-2.0 * gust_length, 0.0, 0.0)),
)

## Generate the simple HALE aircraft

In [ ]:
hale = generate_simple_hale(flowfield=flowfield, sigma_wing=sigma_wing)
n_tstep = int(physical_time / float(hale.aero.dt)) + 1  # number of timesteps for the dynamic solve

## Trim

Balance the forces on the aircraft using thrust and elevator deflection, driving drag, lift, and pitching moment to zero.

In [ ]:
static_sol, trim_vars = hale.trim(
    prescribed_dofs=jnp.arange(6),
    zero_force_dofs=(0, 2, 4),  # balance drag, lift, and pitching moment
    trim_cs="elevator",
    thrust_nodes="thrust",
    trim_orientation="y",
    horseshoe=False,
)

## Initialise dynamic case

Switch from static-aircraft/dynamic-freestream to dynamic-aircraft/static-freestream
for the free-flying gust encounter. Note that there are no prescribed degrees of freedom (empty tuple) for the free aircraft.

In [ ]:
dynamic_init = hale.initialise_dynamic(static_case=static_sol, prescribed_dofs=())

## Dynamic solve

Run the gust encounter for the free aircraft.

In [ ]:
dynamic_sol = hale.dynamic_solve(
    init_case=dynamic_init, prescribed_dofs=(), n_tstep=n_tstep
)

## Post-processing

Extract time histories of the root rigid-body motion and root strains.

In [ ]:
t = jnp.arange(n_tstep) * hale.aero.dt  # time vector

# root node z-displacement (rigid-body heave)
root_z = dynamic_sol.structure.x[:, 0, 2]

# root torsion/bending strains
root_eps = dynamic_sol.structure.eps[:, 0, 3:]

## Plots

### Aircraft rigid-body motion

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, root_z, label="Root (rigid body)")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Vertical position [m]")
ax.set_title("Vertical position")
ax.legend()
plt.show()

### Root strains

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 8))
for i, (ax, label) in enumerate(zip(axes, ["Torsional", "Out-of-plane bending"])):
    ax.plot(t, root_eps[:, i])
    ax.set_title(label)
    ax.set_ylabel("Strain")
axes[-1].set_xlabel("Time [s]")
fig.suptitle("Root strains")
fig.tight_layout()
plt.show()

### VTK output

Optionally write Paraview-compatible VTK files for 3D visualisation.

In [ ]:
# out_paths = dynamic_sol.plot("./simple_hale/")